# Producer 

In [7]:
# 1. Instalar el cliente de Kafka dentro de Jupyter
!pip install kafka-python-ng pandas

import time
import json
import pandas as pd
from kafka import KafkaProducer

# 2. Configurar la conexión. 
# ATENCIÓN: Como estamos dentro de la red de Docker, usamos el listener INTERNAL (puerto 9093)
BOOTSTRAP_SERVERS = 'kafka:9093'
TOPIC_NAME = 'transactions'
CSV_PATH = '../data/creditcard.csv'

producer = KafkaProducer(
    bootstrap_servers=[BOOTSTRAP_SERVERS],
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

print("Cargando dataset...")
df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado. Total de filas a simular: {len(df)}")

print("Enviando datos en streaming (Presiona Stop en Jupyter para parar)...")
try:
    for index, row in df.iterrows():
        transaction = row.to_dict()
        producer.send(TOPIC_NAME, value=transaction)
        
        # Enviamos 10 transacciones por segundo para probar
        time.sleep(0.1) 
        
        if index % 50 == 0 and index > 0:
            print(f">> Enviados {index} eventos a Kafka.")
except KeyboardInterrupt:
    print("\nSimulación detenida.")
finally:
    producer.flush()
    producer.close()

Cargando dataset...
Dataset cargado. Total de filas a simular: 284807
Enviando datos en streaming (Presiona Stop en Jupyter para parar)...
>> Enviados 50 eventos a Kafka.

Simulación detenida.


In [5]:
from os import listdir
CSV_PATH = '/work/data/creditcard.csv'
CSV_PATH = '/'
listdir(CSV_PATH)

['dev',
 'sys',
 'home',
 'etc',
 'lib64',
 'srv',
 'lib32',
 'usr',
 'mnt',
 'run',
 'tmp',
 'sbin',
 'bin',
 'libx32',
 'root',
 'media',
 'proc',
 'lib',
 'opt',
 'boot',
 'var',
 '.dockerenv']

In [4]:
from pyspark.sql import SparkSession
import time

# 1. Inicializar Spark en modo local (Ya lo tienes bien)
spark = SparkSession.builder \
    .appName("TestDeteccionFraude") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# 2. Leer de Kafka con 'earliest' (Ya lo tienes bien)
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Pasar el payload binario a string legible (Ya lo tienes bien)
df_readable = df_kafka.selectExpr("CAST(value AS STRING) as json_payload")

# 4. CAMBIO CRÍTICO: En vez de 'console', guardamos en una tabla en MEMORIA llamada 'tabla_tfm'
query = df_readable.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("tabla_tfm") \
    .start()

# 5. Esperamos 10 segundos para dar tiempo a que Spark recoja los datos del pasado de Kafka
print("Conectando con Kafka y rellenando la tabla en memoria...")
time.sleep(10)

# 6. Forzamos a Jupyter a mostrar el contenido haciendo una consulta SQL a la memoria
print("\n--- ¡DATOS DETECTADOS EN TU PANTALLA! ---")
spark.sql("SELECT * FROM tabla_tfm LIMIT 10").show(truncate=False)

# 7. Detenemos el stream limpiamente
query.stop()
print("\nStream detenido con éxito.")

Conectando con Kafka y rellenando la tabla en memoria...

--- ¡DATOS DETECTADOS EN TU PANTALLA! ---
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_payload                                                                                                         

In [1]:
import pyspark
print(pyspark.__version__)

3.5.0


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.sql.functions import col, from_json
import time

# 1. Inicializar Spark con la versión exacta
spark = SparkSession.builder \
    .appName("TestDeteccionFraude") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# 2. Definir el esquema exacto de tu CSV de Kaggle
schema = StructType([
    StructField("Time", DoubleType(), True),
    *[StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)], # Genera V1 hasta V28
    StructField("Amount", DoubleType(), True),
    StructField("Class", DoubleType(), True) # Class es 0 (normal) o 1 (fraude)
])

# 3. Leer de Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9093") \
    .option("subscribe", "transactions") \
    .option("startingOffsets", "earliest") \
    .load()

# 4. Transformación Mágica: Convertir binario a String y luego desempaquetar el JSON
df_parsed = df_kafka \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), schema).alias("data")) \
    .select("data.*") # Esto extrae todas las variables en columnas separadas

# 5. Guardar en memoria para visualizar
query = df_parsed.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("tabla_tfm") \
    .start()

print("Conectando con Kafka, desempaquetando JSON y rellenando tabla...")
time.sleep(10)

print("\n--- ¡DATOS LISTOS PARA EL MODELO! ---")
spark.sql("SELECT Time, V1, V2, Amount, Class FROM tabla_tfm LIMIT 5").show(truncate=False)

query.stop()
print("\nStream detenido con éxito.")

Conectando con Kafka, desempaquetando JSON y rellenando tabla...

--- ¡DATOS LISTOS PARA EL MODELO! ---
+----+------------------+-------------------+------+-----+
|Time|V1                |V2                 |Amount|Class|
+----+------------------+-------------------+------+-----+
|0.0 |-1.3598071336738  |-0.0727811733098497|149.62|0.0  |
|0.0 |1.19185711131486  |0.26615071205963   |2.69  |0.0  |
|1.0 |-1.35835406159823 |-1.34016307473609  |378.66|0.0  |
|1.0 |-0.966271711572087|-0.185226008082898 |123.5 |0.0  |
|2.0 |-1.15823309349523 |0.877736754848451  |69.99 |0.0  |
+----+------------------+-------------------+------+-----+


Stream detenido con éxito.
